In [1]:
from mpramnist.Gosai2024.dataset import GosaiDataset

from mpramnist.Guo2023.dataset import GuoMultiDataset
from mpramnist.Guo2023.dataset import GuoSingleDataset
from mpramnist.Guo2023.trainer import LitModel_Guo

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import mpramnist.transforms as t

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import lightning.pytorch as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from torchmetrics import PearsonCorrCoef

import pandas as pd

BATCH_SIZE = 1024
NUM_WORKERS = 8

I0000 00:00:1787120248.139650 2747852 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787120249.021909 2747852 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
import warnings
warnings.filterwarnings("ignore")


# Guo SNP Dataset Description

Dataset contains results from Guo *et al* MPRA assay of **2,221 non-coding variants**, associated with ten neuropsychiatric disorderss, 
conducted in several immune and neural cells. 

The assay used **145-bp** SNP-centered itervals for both reference and alternate alleles.


Available cell types in the Dataset

| **Cell Type** | **Description** | **Number of variants tested** | 
| :-----------: | :-----------: | :-----------: |
| AST | Astrocytes | 2088 |
| ES | Embryonic Stem Cell | 2070 |
| N-D2 | ES-derived neural cells, day 2 | 2108 |
| N-D4 | ES-derived neural cells, day 4 | 2112 |
| N-D10 | ES-derived neural cells, day 10 | 2093 |
| A-NPC | Anterior Neural Progenitor Cells | 2122 |
| HEK293T | Immortalized human embryonic kidney cells (control) | 2150 |
| D283 | Medulloblastoma cell line | 1806 |
| D341 | Medulloblastoma cell line | 1812 |
| IMR.prog | Nondifferentiated IMR-32 neuroblastoma cells | 1812 |
| IMR.diff | Differentiated IMR-32 neuroblastoma cells | 1808 |
| SHSY5Y.prog | Nondifferentiated SH-SY5Y neuroblastoma cells | 1810 |
| SHSY5Y.diff | Differentiated SH-SY5Y neuroblastoma cells | 1812 |

The variant was considered as **daSNP** if it achieves | logFC| > 0.05 and an FDR-corrected p-value < 0.05

# Current Workflow

In this notebook, we:

1. Train the **MPRALegNet** model on the **Gosai SK-N-SH dataset**

2. Assess its predictive power using **Guo's SNPs**

## **Train MPRALegNet model using Gosai SK-N-SH data**

In [3]:
cell_types = ["SKNSH"]

# preprocessing
train_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(0.5), t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])

# load the data
train_dataset_own = GosaiDataset(
    split="train",
    transform=train_transform,
    filtration="own",
    cell_types=cell_types,
    stderr_columns=["SKNSH_lfcSE"],  
    stderr_threshold=1.0,
    std_multiple_cut=6.0,
    up_cutoff_move=3.0,
    duplication_cutoff=0.5,
    root="/media/storage/lizzzafomenko/data",
)

# Use the same parameters to valid and test
val_dataset_own = GosaiDataset(split="val", filtration="own", cell_types=cell_types, stderr_columns=["SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root='/media/storage/lizzzafomenko/data',)
test_dataset_own = GosaiDataset(split="test", filtration="own", cell_types=cell_types, stderr_columns=["SKNSH_lfcSE"], stderr_threshold=1.0, std_multiple_cut=6.0, up_cutoff_move=3.0, transform=val_test_transform, root='/media/storage/lizzzafomenko/data',)

print(train_dataset_own)

Dataset GosaiDataset (MpraDaraset)
    Number of datapoints: 842024
    Root location: /media/storage/lizzzafomenko/data/Malinois
    Using split: ['1', '2', '3', '4', '5', '6', '8', '9', '10', '11', '12', '14', '15', '16', '17', '18', '20', '22', 'Y']
    Split: {'train': 668946, 'val': 58809, 'test': 62582}
    Task: Regression
    Description: The Gosai dataset includes 798,064 sequences tested in the K562, HepG2, and SK-N-SH cell lines. The original sequence length is approximately 200 nucleotides, and it is recommended to extend them to 600 bp. The task is to predict three normalized regulatory activity values for the respective cell lines.


In [4]:
# encapsulate data into DataLoader form
train_loader = DataLoader(dataset=train_dataset_own, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(dataset=val_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(dataset=test_dataset_own, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset_own[0][0])
out_channels = len(cell_types)

In [5]:
# initialize LegNet model

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)


seq_model = LitModel_Guo(
    model=model,
    loss=nn.MSELoss(),
    weight_decay=0.1,
    lr=0.01,
    print_each=1,
    lr_scheduler_to_use="onecycle",
    learning_rate_reduce=True
)

In [6]:
checkpoint_callback = ModelCheckpoint(
    monitor="val_pearson", mode="max", save_top_k=1, save_last=False
)

trainer = L.Trainer(
    accelerator="gpu",
    devices=[3],
    precision="16-mixed",
    enable_progress_bar=True,
    max_epochs=1,
    callbacks=[checkpoint_callback],
)

trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA L40S') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ HumanLegNet     │  1.3 M │ train │     0 │
│ 1 │ loss          │ MSELoss         │      0 │ train │     0 │
│ 2 │ train_pearson │ PearsonCorrCoef │      0 │ train │     0 │
│ 3 │ val_pearson   │ PearsonCorrCoef │      0 │ train │     0 │
│ 4 │ test_pearson  │ PearsonCorrCoef │      0 │ train │     0 │
└───┴───────────────┴─────────────────┴────────┴───────┴───────┘

Trainable params: 1.3 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.3 M                                                                                                
Total estimated model params size (MB): 5.290                                                                      
Modules in train mode: 120                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

---------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 1.64249 | Val Pearson: 0.24287 | Train Pearson: nan 
---------------------------------------------------------------------------

...

Metric val_loss improved. New best score: 1.539
`Trainer.fit` stopped: `max_epochs=1` reached.


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    1.0058424472808838     │
│       test_pearson        │    0.47435060143470764    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 1.0058424472808838, 'test_pearson': 0.47435060143470764}]

In [7]:
# load model from the best checkpoint

best_model_path = checkpoint_callback.best_model_path
seq_model = LitModel_Guo.load_from_checkpoint(best_model_path, model=model, loss=nn.MSELoss(), weight_decay=0.1, lr=0.01, print_each=1, use_one_cycle = True,)

## **Evaluate MPRALegNet model using Guo SNPs Data**

Initialize transformations for both `forward` and `reverse_complement` sequences.

For each sequence add flanks from Gosai assay and crop to the length of 600 base pairs.

In [8]:
forw_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.Seq2Tensor()])
revcomp_transform = t.Compose([t.AddFlanks(GosaiDataset.LEFT_FLANK, GosaiDataset.RIGHT_FLANK), t.CenterCrop(600), t.ReverseComplement(1), t.Seq2Tensor()])

#### **GuoMultiDataset**

download full Guo *et all* MPRA data with `GuoMultiDataset`

In [10]:
# Use all available cell types
CELL_TYPES = ['AST', 'ES', 'N-D2', 'N-D4', 'N-D10', 'A-NPC', 'D283', 'D341', 'IMR.diff', 'IMR.prog', 'SHSY5Y.diff', 'SHSY5Y.prog', 'HEK293T']


# initialize datasets and encapsulate data into DataLoader form
predict_forward_dataset = GuoMultiDataset(split = 'test', length = 145, cell_types = CELL_TYPES, transform=forw_transform, root = '/media/storage/lizzzafomenko/data')
predict_forward_dataloader = DataLoader(dataset=predict_forward_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

predict_revcomp_dataset = GuoMultiDataset(split = 'test', length = 145, cell_types = CELL_TYPES, transform=revcomp_transform, root = '/media/storage/lizzzafomenko/data')
predict_revcomp_dataloader = DataLoader(dataset=predict_revcomp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [11]:
# get model predictions

forw_preds = trainer.predict(seq_model, dataloaders=predict_forward_dataloader)
revcomp_preds = trainer.predict(seq_model, dataloaders=predict_revcomp_dataloader)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Output()

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Output()

Test model performance using `Guo_variants_prediction` function, which:
1.  processes model predictions for `ref` and `alt` alleles
2.  calculates `variant scores` as `alt_pred - ref_pred`
3.  calculates `pearson correlation` between model predictins and MPRA variant scores
4.  returns a `pd.DataFrame` with results for all cell types or print all correlation results

In [19]:
def Guo_variants_prediction(forw_preds, revcomp_preds, cell_types, return_df = True):
    """
    Calculate Pearson correlation between model-predicted and MPRA-measured 
    variant effects for each cell type.
    
    Parameters
    ----------
    forw_preds : list of dict
        List of dictionaries containing model predictions for forward sequences.
        Each dict must have keys: 'target', 'fdr', 'ref_predicted', 'alt_predicted'
    revcomp_preds : list of dict
        List of dictionaries containing model predictions for reverse complement sequences.
        Same structure as forw_preds
    cell_types : list of str
        List of cell type names corresponding to columns in target/fdr tensors
    return_df : bool
        If True, return results as pd.DataFrame; otherwise print them
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - cell_type: name of the cell type
        - n_all: number of variants tested in this cell type
        - pearsonr_all: Pearson correlation across all tested variants
        - n_daSNP: number of significant variants (FDR < 0.05)
        - pearsonr_daSNP: Pearson correlation across significant variants only
    """

    targets = torch.cat([pred["target"] for pred in forw_preds])
    fdrs = torch.cat([pred["fdr"] for pred in forw_preds])

    if len(targets.shape) == 1:
        targets = targets.reshape(-1, 1)
        fdrs = fdrs.reshape(-1, 1)

    # Forward sequence prediction
    y_preds_forw_ref = torch.cat([pred["ref_predicted"] for pred in forw_preds])
    y_preds_forw_alt = torch.cat([pred["alt_predicted"] for pred in forw_preds])

    # Reverse-complement complement predictions
    y_preds_revcomp_ref = torch.cat([pred["ref_predicted"] for pred in revcomp_preds])
    y_preds_revcomp_alt = torch.cat([pred["alt_predicted"] for pred in revcomp_preds])

    # Average forward and reverse complement predictions for both ref and alt alleles
    y_preds_ref = torch.mean(torch.stack([y_preds_forw_ref, y_preds_revcomp_ref]), dim=0)
    y_preds_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_revcomp_alt]), dim=0)

    # Variant score, predicted by model
    variant_prediction = y_preds_alt - y_preds_ref
    
    results = []
    pears = PearsonCorrCoef()

    for i in range(len(cell_types)):

        # All variants tested in this cell type
        mask_all = ~torch.isnan(targets[:, i].squeeze())                                      
        pearsonr_all = pears(variant_prediction.squeeze()[mask_all], targets[:, i].squeeze()[mask_all])

        # Significant variants for the cell type only (FDR < 0.05)
        mask_daSNP = mask_all & (fdrs[:, i].squeeze() < 0.05)
        pearsonr_daSNP = pears(variant_prediction.squeeze()[mask_daSNP], targets[:, i].squeeze()[mask_daSNP])

        if return_df:
            results.append({
                'cell_type': cell_types[i],
                'n_all': mask_all.sum().item(),
                'pearsonr_all': pearsonr_all.item(),
                'n_daSNP': mask_daSNP.sum().item(),
                'pearsonr_daSNP': pearsonr_daSNP.item()
            })
        
        else:
            print(f'Pearson correlation for {cell_types[i]}')
            print(f'\t\t all SNPs (n = {mask_all.sum().item()}): {pearsonr_all.item():.6f}')
            print(f'\t\t daSNPs (n = {mask_daSNP.sum().item()}): {pearsonr_daSNP.item():.6f}')

    if return_df:
        df = pd.DataFrame(results)
        return df


In [13]:
Guo_variants_prediction(forw_preds, revcomp_preds, CELL_TYPES)

,cell_type,n_all,pearsonr_all,n_daSNP,pearsonr_daSNP
0,AST,2087,0.039415,65,-0.063662
1,ES,2069,0.004748,104,0.075987
2,N-D2,2107,-0.049126,84,-0.328271
3,N-D4,2111,0.026674,186,0.003132
4,N-D10,2092,-0.021816,149,-0.090446
5,A-NPC,2121,-0.017999,71,0.049916
6,D283,1779,-0.039230,70,-0.141417
7,D341,1787,-0.045070,150,-0.069971
8,IMR.diff,1783,-0.040048,188,-0.133068
9,IMR.prog,1789,-0.059293,269,-0.029242


#### **GuoSingleDataset**

In [15]:
# Use GuoSingleDataset when interested in the only cell type

CELL_TYPE = 'SHSY5Y.diff'

predict_forward_dataset = GuoSingleDataset(split = 'test', length = 145, cell_type = CELL_TYPE, transform=forw_transform, root = '/media/storage/lizzzafomenko/data')
predict_forward_dataloader = DataLoader(dataset=predict_forward_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

predict_revcomp_dataset = GuoSingleDataset(split = 'test', length = 145, cell_type = CELL_TYPE, transform=revcomp_transform, root = '/media/storage/lizzzafomenko/data')
predict_revcomp_dataloader = DataLoader(dataset=predict_revcomp_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [16]:
# encapsulate data into DataLoader form

forw_preds = trainer.predict(seq_model, dataloaders=predict_forward_dataloader)
revcomp_preds = trainer.predict(seq_model, dataloaders=predict_revcomp_dataloader)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Output()

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Output()

In [20]:
Guo_variants_prediction(forw_preds, revcomp_preds, [CELL_TYPE], False)

Pearson correlation for SHSY5Y.diff
		 all SNPs (n = 1790): -0.029211
		 daSNPs (n = 173): -0.040829


## Test AlphaGenome model

In [ ]:
from mpramnist.models import predict_variants_AlphaGenome, filter_tracks

I0000 00:00:1782313608.027665 3158147 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782313608.950184 3158147 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [4]:
# Use all available cell types
CELL_TYPES = ['AST', 'ES', 'N-D2', 'N-D4', 'N-D10', 'A-NPC', 'D283', 'D341', 'IMR.diff', 'IMR.prog', 'SHSY5Y.diff', 'SHSY5Y.prog', 'HEK293T']

basic_transform = t.Compose([t.Seq2Tensor()])
ag_dataset = GuoMultiDataset(split = 'test', length = 2**11, cell_types = CELL_TYPES, transform=basic_transform, root = '/media/storage/lizzzafomenko/data')


In [5]:
ag_preds = predict_variants_AlphaGenome(weights_path = '/media/storage/lizzzafomenko/data/AlphaGenome/weights/model_all_folds.safetensors', 
                            dataset = ag_dataset, 
                            batch_size = 4, 
                            device = 'cuda:1') 


In [18]:
def Guo_AlphaGenome_variants_prediction(preds, cell_types, return_df = True, biosample_names = None, exact_match = False):
    """
    Calculate Pearson correlation between model-predicted and MPRA-measured 
    variant effects for each cell type.
    
    Parameters
    ----------
    preds : list of dict
        List of dictionaries containing model predictions.
        Each dict must have keys: 'target', 'fdr', 'ref_predicted', 'alt_predicted'
    cell_types : list of str
        List of cell type names corresponding to columns in target/fdr tensors
    biosample_names : list[list[str]]
        List of biosample names to be used to filter AlphaGenome tracks 
        before variant prediction. If None, all tracks will be used for prediction
    return_df : bool
        If True, return results as pd.DataFrame; otherwise print them
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - cell_type: name of the cell type
        - n_all: number of variants tested in this cell type
        - pearsonr_all: Pearson correlation across all tested variants
        - n_daSNP: number of significant variants (FDR < 0.05)
        - pearsonr_daSNP: Pearson correlation across significant variants only
    """

    targets = torch.cat([pred["target"] for pred in preds])
    fdrs = torch.cat([pred["fdr"] for pred in preds])

    # Forward sequence prediction
    y_preds_ref = torch.cat([pred["ref_predicted"] for pred in preds])
    y_preds_alt = torch.cat([pred["alt_predicted"] for pred in preds])

    # Variant score, predicted by model
    variant_prediction = y_preds_alt - y_preds_ref
    
    results = []
    pears = PearsonCorrCoef()


    for i in range(len(cell_types)):

        if biosample_names:
            var_preds = filter_tracks(variant_prediction, biosample_names[i], exact_match = exact_match)
            var_preds = var_preds.mean(axis = 1)
        else:
            var_preds = variant_prediction.mean(axis = 1)

        # All variants tested in this cell type
        mask_all = ~torch.isnan(targets[:, i].squeeze())                                      
        pearsonr_all = pears(var_preds.squeeze()[mask_all], targets[:, i].squeeze()[mask_all])

        # Significant variants for the cell type only (FDR < 0.05)
        mask_daSNP = mask_all & (fdrs[:, i].squeeze() < 0.05)
        pearsonr_daSNP = pears(var_preds.squeeze()[mask_daSNP], targets[:, i].squeeze()[mask_daSNP])

        if return_df:
            results.append({
                'cell_type': cell_types[i],
                'n_all': mask_all.sum().item(),
                'pearsonr_all': pearsonr_all.item(),
                'n_daSNP': mask_daSNP.sum().item(),
                'pearsonr_daSNP': pearsonr_daSNP.item()
            })
        
        else:
            print(f'Pearson correlation for {cell_types[i]}')
            print(f'\t\t all SNPs (n = {mask_all.sum().item()}): {pearsonr_all.item():.6f}')
            print(f'\t\t daSNPs (n = {mask_daSNP.sum().item()}): {pearsonr_daSNP.item():.6f}')

    if return_df:
        df = pd.DataFrame(results)
        return df


In [19]:
# use AlphaGenome with defined tracks

Guo_AlphaGenome_variants_prediction(ag_preds,  CELL_TYPES, True,
    [['astrocyte'], ['neuronal stem cell'], ['neuronal stem cell'], ['neuronal stem cell'], ['neuronal stem cell'], ['neuronal stem cell'], ['D721Med'], ['D721Med'], ['IMR-90'], ['IMR-90'], ['SK-N-SH'], ['SK-N-SH'], ['HEK293']], 
                        False)

,cell_type,n_all,pearsonr_all,n_daSNP,pearsonr_daSNP
0,AST,2087,-0.029421,65,-0.145215
1,ES,2069,-0.008477,104,-0.137364
2,N-D2,2107,0.004361,84,0.015778
3,N-D4,2111,-0.002302,186,-0.029444
4,N-D10,2092,-0.022203,149,-0.038650
5,A-NPC,2121,-0.008538,71,0.059386
6,D283,1779,0.018110,70,0.052371
7,D341,1787,0.002618,150,-0.001008
8,IMR.diff,1783,-0.010485,188,0.023330
9,IMR.prog,1789,0.009425,269,0.096482


In [20]:
# use mean of all tracks

Guo_AlphaGenome_variants_prediction(ag_preds,  CELL_TYPES)

,cell_type,n_all,pearsonr_all,n_daSNP,pearsonr_daSNP
0,AST,2087,-0.027020,65,-0.165269
1,ES,2069,-0.021161,104,-0.185946
2,N-D2,2107,-0.001916,84,0.029429
3,N-D4,2111,-0.015788,186,-0.067027
4,N-D10,2092,-0.029462,149,-0.095506
5,A-NPC,2121,-0.017467,71,-0.049406
6,D283,1779,0.043212,70,0.224802
7,D341,1787,0.009298,150,0.148850
8,IMR.diff,1783,0.003698,188,0.074434
9,IMR.prog,1789,0.003828,269,0.136624
